# nb10.1 - Romano-Wolf Step-Down: When Twenty Outcomes Are Really One Question

*Week 10, Chapter 10.2 companion. Priya comes back from the symposium with twenty climate-risk outcomes
she ran in panic the night before her poster session. A discussant raised a hand and asked,
"How are you correcting for multiplicity?" She had no answer. This notebook gives her one.*

The reveal-the-trick version of the lesson: **if you test twenty hypotheses at the 5% level on outcomes
that are correlated, the probability that at least one of your twenty p-values lands below 0.05 by
sheer chance is much higher than 5%.** That number - the probability of *any* false rejection across
the whole family - is the **family-wise error rate (FWER)**. The Romano-Wolf step-down procedure
controls FWER by **resampling** the joint null distribution of all twenty test statistics, so it
respects the **correlation** across outcomes instead of pretending they are independent. The price you
pay (a small loss of power versus a single-outcome test) is small. The naive Bonferroni you learned in
intro stats also controls FWER, but it ignores correlation and is therefore overconservative when, as
in finance, the outcomes move together. The BH-FDR you saw in chapter 10.2 controls a *different*
quantity (the **expected false-discovery rate** among the rejected) and tends to reject more - useful
when the science is exploratory, dangerous when a single false claim breaks a paper.

By the end of this notebook you will see, in three plots and one table, why the three corrections
disagree, and you will have a hand-rolled Romano-Wolf routine you can drop into your own paper.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless: render to buffers, never to a screen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

rng = np.random.default_rng(42)  # one pinned generator drives the whole notebook

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.float_format", lambda v: f"{v:0.4f}")

## 1. A synthetic 20-outcome DGP with realistic correlation

The honest way to study a multiple-testing procedure is to test it on data where you **know the truth**.
We build a panel of $N=400$ firm-quarters. Each firm has a binary treatment $D_i$ (e.g., a new
climate-disclosure rule applies to it or not) and a baseline covariate $X_i$. We then construct
$K=20$ outcomes: think of them as twenty different climate-risk measures (carbon intensity, supply-chain
exposure, physical-risk score, transition-risk score, and so on).

Two design choices make this realistic:

1. **Most outcomes are nulls.** Only the first three outcomes are truly affected by the treatment
   ($\tau_k = 0.18$ for $k\in\{0,1,2\}$); the remaining seventeen have $\tau_k = 0$. This mirrors a
   realistic empirical situation: you have one or two true effects buried in a forest of nulls.

2. **The outcomes are correlated.** We draw the regression errors from a multivariate normal with a
   shared "industry shock" loading. Two outcomes that share the same shock will move together, so
   their t-statistics under the null are correlated. This is exactly the situation in which Bonferroni
   is too conservative and Romano-Wolf earns its keep.

In [ ]:
def make_dgp(rng, N=400, K=20, tau=None, rho=0.6):
    # Synthetic DGP. Returns DataFrame with treat, X, and K outcomes y0..y{K-1}.
    # The shared shock 'eta' (loading rho) induces correlation across outcomes.
    # The first three outcomes have a real treatment effect; the rest are nulls.
    if tau is None:
        tau = np.array([0.18, 0.18, 0.18] + [0.0] * (K - 3))
    treat = rng.binomial(1, 0.5, size=N)
    X = rng.normal(0, 1, size=N)
    eta = rng.normal(0, 1, size=N)              # shared latent shock (industry-level)
    eps = rng.normal(0, 1, size=(N, K))         # idiosyncratic outcome-specific noise

    # Each outcome loads on the shared shock with weight rho and on its own noise with sqrt(1-rho^2).
    loading = np.sqrt(1 - rho ** 2)
    Y = (tau[None, :] * treat[:, None]
         + 0.4 * X[:, None]
         + rho * eta[:, None]
         + loading * eps)

    df = pd.DataFrame(Y, columns=[f"y{k}" for k in range(K)])
    df["treat"] = treat
    df["X"] = X
    return df, tau

df, tau_true = make_dgp(rng)
print(df.shape)
print("truly nonzero effects at indices:", np.where(tau_true != 0)[0].tolist())
df.head()

**What the DGP gives us.** Twenty outcomes; three real treatment effects (`y0`, `y1`, `y2`) and
seventeen nulls. The shared shock `eta` with loading $\rho=0.6$ makes the outcomes correlated under
the null - which is the whole reason we are doing this. Let's eyeball the correlation matrix so we
trust the simulation.

In [ ]:
corr = df[[f"y{k}" for k in range(20)]].corr().values
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, vmin=-0.1, vmax=1.0, cmap="viridis")
ax.set_title("Correlation matrix of the 20 outcomes")
ax.set_xlabel("outcome k"); ax.set_ylabel("outcome k")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.close(fig)
print(f"average off-diagonal correlation = {(corr.sum() - 20) / (20*19):0.3f}")

The average off-diagonal correlation should sit near $\rho^2 = 0.36$, which is exactly the
regime in which **Bonferroni overcorrects**. Bonferroni assumes the worst case (independence-or-positive
dependence with no structure) and divides $\alpha$ by $K$. When the t-statistics are positively
correlated, fewer "independent shots at significance" are really being taken, so the true FWER under
the null is well below the Bonferroni budget.

## 2. Twenty regressions, twenty t-statistics

For each outcome $k$ we run a simple OLS:
$$y_{ik} = \alpha_k + \tau_k D_i + \gamma_k X_i + u_{ik}.$$
We collect the t-statistic $t_k = \hat\tau_k / \widehat{\mathrm{SE}}(\hat\tau_k)$ and the p-value.
These are the twenty numbers a discussant would point at in your draft table.

In [ ]:
def run_regressions(df, K=20):
    # Run K OLS regressions; return arrays of coefficients, std errors, t-stats, p-values.
    Z = sm.add_constant(df[["treat", "X"]].values)  # [1, D, X]
    coefs = np.zeros(K); ses = np.zeros(K); tstats = np.zeros(K); pvals = np.zeros(K)
    for k in range(K):
        y = df[f"y{k}"].values
        res = sm.OLS(y, Z).fit()
        coefs[k] = res.params[1]
        ses[k] = res.bse[1]
        tstats[k] = res.tvalues[1]
        pvals[k] = res.pvalues[1]
    return coefs, ses, tstats, pvals

coefs, ses, tstats, pvals = run_regressions(df)
report = pd.DataFrame({"k": np.arange(20), "tau_true": tau_true,
                       "tau_hat": coefs, "se": ses, "t": tstats, "p_raw": pvals})
report

**Read the table.** Outcomes 0, 1, 2 have true effects and tend to come back with small p-values.
Several of the nulls will also have raw p-values below 0.05 by chance - that is precisely the false
rejection we are trying to control. Let us count them.

In [ ]:
raw_rejects = (pvals < 0.05)
print(f"raw rejections at alpha=0.05: {raw_rejects.sum()} out of 20")
print(f"  - true positives (k in 0,1,2): {raw_rejects[:3].sum()}")
print(f"  - false positives (k in 3..19): {raw_rejects[3:].sum()}")

## 3. Three corrections in one place

We now apply three FWER/FDR corrections to the same vector of p-values.

**Bonferroni.** Reject the $k$-th hypothesis iff $p_k \le \alpha/K$. Controls FWER under any dependence
structure - at the price of severe conservatism when outcomes are correlated.

**Benjamini-Hochberg FDR.** Sort p-values $p_{(1)} \le \cdots \le p_{(K)}$. Find the largest $k$ such
that $p_{(k)} \le k \alpha / K$. Reject all hypotheses with rank $\le k$. Controls the *expected*
false-discovery rate at level $\alpha$ - not FWER. More rejections, but a different guarantee.

**Romano-Wolf step-down.** Approximate the joint null distribution of $|t_1|, \dots, |t_K|$ by
**bootstrapping** (or, here, by simulating directly from the estimated residual covariance). Step
down: at each step, find the *current maximum* $|t|$, compute its bootstrap p-value against the
joint maximum, and only proceed to test the next-largest if the current one rejects. This respects
the empirical correlation and is **uniformly more powerful than Bonferroni** while still controlling
FWER.

In [ ]:
def bonferroni(pvals, alpha=0.05):
    K = len(pvals)
    return pvals <= alpha / K

def bh_fdr(pvals, alpha=0.05):
    K = len(pvals); order = np.argsort(pvals); sorted_p = pvals[order]
    crit = (np.arange(1, K + 1) / K) * alpha
    below = sorted_p <= crit
    if not below.any():
        return np.zeros(K, dtype=bool)
    cutoff_rank = np.where(below)[0].max()
    rejected_sorted = np.zeros(K, dtype=bool); rejected_sorted[:cutoff_rank + 1] = True
    out = np.zeros(K, dtype=bool); out[order] = rejected_sorted
    return out

print("Bonferroni rejects:", np.where(bonferroni(pvals))[0].tolist())
print("BH-FDR  rejects   :", np.where(bh_fdr(pvals))[0].tolist())

**The hand-rolled Romano-Wolf step-down.** We need a joint null distribution of the t-statistics.
The cleanest finite-sample way to get one is the **wild bootstrap**: hold the regressors fixed; resample
the residuals with random sign flips; re-run all $K$ regressions. Repeat $B$ times. At each replication
we collect the *vector* of t-statistics, then the *maximum absolute t* across the currently active
hypotheses. The step-down works from the largest observed $|t|$ down to the smallest, peeling off
hypotheses as it goes - see Romano & Wolf (2005a, *Econometrica* 73(4):1237-1282) for the formal
algorithm.

In [ ]:
def romano_wolf_stepdown(df, K=20, B=1000, alpha=0.05, seed=123):
    # Hand-rolled Romano-Wolf step-down using a wild-bootstrap (Rademacher) null distribution.
    # Returns boolean array of length K marking which hypotheses are rejected at FWER alpha.
    boot_rng = np.random.default_rng(seed)
    n = len(df)
    Z = sm.add_constant(df[["treat", "X"]].values)
    obs_t = np.zeros(K)
    null_resid = np.zeros((n, K))      # residuals from the *restricted* (no-treat) fit
    fitted = np.zeros((n, K))          # fitted values from the *restricted* fit
    Zr = sm.add_constant(df[["X"]].values)
    for k in range(K):
        y = df[f"y{k}"].values
        res_full = sm.OLS(y, Z).fit()
        obs_t[k] = abs(res_full.tvalues[1])
        res_restr = sm.OLS(y, Zr).fit()       # impose H0: tau_k = 0
        fitted[:, k] = res_restr.fittedvalues
        null_resid[:, k] = res_restr.resid
    abs_obs = obs_t.copy()

    boot_max = np.zeros((B, K))  # row b = |t| vector at replication b
    for b in range(B):
        signs = boot_rng.choice([-1.0, 1.0], size=n)
        boot_t = np.zeros(K)
        for k in range(K):
            y_star = fitted[:, k] + signs * null_resid[:, k]
            res_b = sm.OLS(y_star, Z).fit()
            boot_t[k] = abs(res_b.tvalues[1])
        boot_max[b] = boot_t

    order_desc = np.argsort(-abs_obs)            # indices from largest |t| to smallest
    rejected = np.zeros(K, dtype=bool)
    for j, k_star in enumerate(order_desc):
        idx_active = order_desc[j:]              # remaining (un-rejected) hypotheses
        boot_max_active = boot_max[:, idx_active].max(axis=1)
        p_rw = (boot_max_active >= abs_obs[k_star]).mean()
        if p_rw <= alpha:
            rejected[k_star] = True
        else:
            break                                # step-down: stop on first non-rejection
    return rejected, abs_obs, boot_max

rw_reject, abs_obs, boot_max = romano_wolf_stepdown(df, B=800)
print("Romano-Wolf rejects:", np.where(rw_reject)[0].tolist())

**What the algorithm did.** It looked at the largest observed $|t|$ among the twenty outcomes,
asked "how often does the bootstrap **maximum** under the joint null beat this number?", and used that
*joint* p-value to decide. Only if the largest survives does it move to the second-largest, and only
the *remaining* hypotheses contribute to the joint distribution at the next step. This is why step-down
beats single-step Bonferroni: at the second step you are comparing against the maximum of $K-1$, not
$K$, draws. Power compounds across steps.

## 4. Side-by-side: who rejects what?

In [ ]:
summary = pd.DataFrame({
    "k": np.arange(20),
    "true_effect": tau_true,
    "p_raw": pvals,
    "raw_05": (pvals < 0.05),
    "bonferroni": bonferroni(pvals),
    "bh_fdr": bh_fdr(pvals),
    "romano_wolf": rw_reject,
})
summary

**Read the columns.** The raw-p column shows the unadjusted picture. Bonferroni is the strictest
of the three FWER procedures, and it will sometimes miss a true effect (a **false negative**) because
the threshold $0.05/20 = 0.0025$ is severe. BH-FDR is the most permissive: it rejects the largest set
because it controls a different (expected proportion) quantity. Romano-Wolf typically sits between
Bonferroni and BH on this DGP - recovering most of the true positives without admitting nulls.

## 5. The simulation that actually settles the argument

A single realization could be lucky. We now repeat the whole experiment $M=200$ times, each draw with a
fresh seed, and tabulate two things at the family level:

- **Family-wise error rate (FWER).** Fraction of simulations in which **at least one** true null is
  rejected. Should be $\le \alpha=0.05$ for a procedure that claims to control FWER.
- **Average power.** Among the three real effects, the fraction correctly rejected - averaged across
  simulations. Higher is better.

If Romano-Wolf does what it advertises, we should see FWER near (or below) $0.05$ and **average power
strictly higher than Bonferroni's** in this correlated-outcome world. We use a small Monte Carlo here
(M=80, B=200) to keep the notebook quick; on a real paper push both up.

In [ ]:
def one_sim(seed, B=200):
    rng_s = np.random.default_rng(seed)
    df_s, tau_s = make_dgp(rng_s)
    _, _, _, p_s = run_regressions(df_s)
    bon = bonferroni(p_s)
    bh  = bh_fdr(p_s)
    rw, _, _  = romano_wolf_stepdown(df_s, B=B, seed=seed + 9999)
    null_idx = np.where(tau_s == 0)[0]
    true_idx = np.where(tau_s != 0)[0]
    def metrics(rej):
        any_false = rej[null_idx].any()
        power = rej[true_idx].mean()
        return any_false, power
    return (metrics(bon), metrics(bh), metrics(rw))

M = 80
out = {"Bonferroni": [], "BH-FDR": [], "Romano-Wolf": []}
for m in range(M):
    bon_m, bh_m, rw_m = one_sim(seed=1000 + m, B=150)
    out["Bonferroni"].append(bon_m)
    out["BH-FDR"].append(bh_m)
    out["Romano-Wolf"].append(rw_m)

agg = {}
for name, lst in out.items():
    arr = np.array(lst)  # (M,2): [any_false, power]
    agg[name] = {"FWER": arr[:, 0].mean(), "AvgPower": arr[:, 1].mean()}
pd.DataFrame(agg).T

**Reading the simulation table.** The FWER column should be at-or-below 5% for Bonferroni and
Romano-Wolf, and **noticeably above 5% for BH-FDR** - because BH controls expected FDR, not FWER, and
will fail FWER by design when many true alternatives are present. The power column tells you the cost
of conservatism: Bonferroni typically loses several percentage points of power to Romano-Wolf when
outcomes are correlated. That is the headline finding of Romano & Wolf (2005a, b) and the practical
reason your discussant asked the question.

## 6. One picture that ties the lesson together

In [ ]:
counts = {
    "Truth (3 real effects)": 3,
    "Raw p<0.05":  int((pvals < 0.05).sum()),
    "Bonferroni":  int(bonferroni(pvals).sum()),
    "BH-FDR":      int(bh_fdr(pvals).sum()),
    "Romano-Wolf": int(rw_reject.sum()),
}
fig, ax = plt.subplots()
ax.bar(list(counts.keys()), list(counts.values()),
       color=["#444", "#c0392b", "#2980b9", "#27ae60", "#8e44ad"])
ax.set_ylabel("rejections at alpha = 0.05")
ax.set_title("How many hypotheses each procedure rejects, on the same 20 outcomes")
for i, v in enumerate(counts.values()):
    ax.text(i, v + 0.05, str(v), ha="center", fontsize=11)
plt.xticks(rotation=15)
plt.tight_layout(); plt.close(fig)
counts

## 7. When Romano-Wolf fails (be honest)

Three failure modes are worth naming:

1. **Tiny samples.** The bootstrap distribution of $|t|$ is only as good as the residual approximation.
   If $N$ is small (think $<50$), the wild bootstrap can have noticeable size distortion. Use the
   pairs bootstrap as a robustness check.

2. **Heavy clustering.** If errors cluster (firm or industry), you must resample at the cluster level
   - sign-flip a whole cluster's residuals together. Failing to do this **understates** the joint null
   variance and lets too many hypotheses through.

3. **A pre-screened family.** If you ran 100 outcomes, picked the 20 that looked promising, and then
   ran Romano-Wolf on those 20, you have *not* controlled FWER over the original 100. The honest move
   is to pre-register the family of outcomes - chapter 10.3 returns to this.

## 8. Your turn

1. **Stress-test correlation.** Re-run the DGP with `rho = 0.0` (independent outcomes) and `rho = 0.9`
   (very correlated). At which $\rho$ does Bonferroni's power gap to Romano-Wolf become severe?
2. **All-null world.** Set `tau = np.zeros(20)` and confirm empirical FWER hovers near nominal $\alpha$
   for both Bonferroni and Romano-Wolf, but BH-FDR's *FWER* (not FDR) drifts upward as $K$ grows.
3. **Sample-size scaling.** Repeat with $N=100$ and $N=1600$. How does Romano-Wolf's power gap to
   Bonferroni scale with $N$?
4. **Cluster bootstrap (harder).** Modify the wild bootstrap so the sign flips are applied at the
   firm level rather than the row level. Compare FWER when outcomes share a firm shock.

**Citations to anchor your writeup.** Romano & Wolf (2005a, *Econometrica* 73(4):1237-1282); Romano &
Wolf (2005b, *Journal of the American Statistical Association* 100(469):94-108); Benjamini & Hochberg
(1995, *JRSS-B* 57(1):289-300).